# Projeto de Ciência de Dados Parceria Semantix

## Predição de inadimplência de clientes

Este projeto investiga como dados financeiros e
comportamentais podem antecipar o risco de inadimplência
no próximo mês.

A proposta está alinhada a um problema real de negócio, a previsão de inadimplência em clientes de crédito.
Antecipar o risco permite decisões mais assertivas, como ajuste de limites, concessão de crédito e estratégias de cobrança.
Isso impacta diretamente a redução de perdas financeiras e melhora a gestão da carteira de clientes.

Portanto, o uso de dados e modelos preditivos se torna essencial para suportar decisões estratégicas.


# Contexto do problema

O objetivo não é só classificar clientes como inadimplentes, mas entender quais comportamentos aumentam esse risco.

Para isso, foi utilizado o dataset público "Default of Credit Card Clients" da UCI, escolhido por representar um cenário real de crédito
e possuir variáveis relevantes de comportamento financeiro.

O projeto foi estruturado em etapas nas seguintes etapas:
problema, dados, análise exploratória, tratamento, modelagem, validação e conclusão.


## 5. Abordagem Analítica 

 Ao longo deste notebook, a análise buscará responder às seguintes perguntas: 
- Quais padrões aparecem com maior frequência entre clientes inadimplentes?
- Quais variáveis parecem ter maior relação com a inadimplência?
- Qual modelo apresenta melhor capacidade preditiva?
- O desempenho observado é consistente quando avaliado com validação cruzada?


In [2]:
# =========================================================
# 1. BIBLIOTECAS
# =========================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")

# Tentativa de importar XGBoost
xgb_disponivel = True
try:
    from xgboost import XGBClassifier
except ImportError:
    xgb_disponivel = False
    print("\n[AVISO] XGBoost não encontrado no ambiente.")
    print("Para incluir o modelo XGBoost, instale com: pip install xgboost\n")

# Configurações visuais
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Bibliotecas e estratégia analítica

As bibliotecas foram organizadas por etapa do pipeline, separando análise de dados, visualização, pré-processamento, modelagem e avaliação.

Foram usadas ferramentas como pandas e numpy para manipulação, seaborn e matplotlib para análise visual e scikit-learn para construção dos pipelines e validação dos modelos.

Além disso, diferentes algoritmos foram aplicados para permitir uma comparação consistente de desempenho.
Essa organização perimite um fluxo mais claro, reproduzível e fácil de entender.


In [ ]:
# =========================================================
# 2. CARREGAMENTO DO DATASET
# =========================================================

CAMINHO_LOCAL = "default_credit_card_clients.csv"

def carregar_dataset():
    try:
        print("=" * 70)
        print("TENTANDO CARREGAR DATASET ONLINE")
        print("=" * 70)

        dataset_obj = fetch_ucirepo(id=350)
        X = dataset_obj.data.features
        y = dataset_obj.data.targets

        df_local = pd.concat([X, y], axis=1)
        df_local.to_csv(CAMINHO_LOCAL, index=False)

        print("Dataset carregado online com sucesso.")
        print(f"Cópia local salva em: {CAMINHO_LOCAL}")

        return df_local, dataset_obj

    except Exception as e:
        print("\nFalha ao carregar online:", e)

        if os.path.exists(CAMINHO_LOCAL):
            print("Carregando dataset a partir do arquivo local...")
            df_local = pd.read_csv(CAMINHO_LOCAL)
            print("Dataset carregado localmente com sucesso.")
            return df_local, None
        else:
            raise FileNotFoundError(
                f"Não foi possível carregar o dataset online e o arquivo "
                f"'{CAMINHO_LOCAL}' não foi encontrado."
            )

df, dataset = carregar_dataset()

# Fonte de dados

A base foi carregada da UCI Machine Learning Repository, atendendo ao requisito de uso de dados públicos e não confidenciais.

Trata-se de um dataset de risco de crédito, com 30 mil registros e diversas variáveis financeiras e comportamentais, como histórico de pagamento, faturas e valores pagos.

Essa base foi escolhida por representar bem um cenário real de análise de inadimplência, além de ser amplamente utilizada na literatura, o que facilita comparação e validação dos resultados.

Por se tratar de um problema com variável alvo binária, ela é adequada para aplicação de modelos de classificação supervisionada.


In [ ]:
# =========================================================
# 3. METADATA E DESCRIÇÃO DAS VARIÁVEIS
# =========================================================

print("\n" + "=" * 70)
print("METADATA DO DATASET")
print("=" * 70)
if dataset is not None:
    print(dataset.metadata)
else:
    print("Metadata indisponível no modo offline.")

print("\n" + "=" * 70)
print("INFORMAÇÕES DAS VARIÁVEIS")
print("=" * 70)
if dataset is not None:
    print(dataset.variables)
else:
    print("Descrição das variáveis indisponível no modo offline.")

# Entendimento do dataset

Antes da modelagem, foi feito o entendimento das variáveis com base na documentação oficial da base.
Isso permitiu interpretar corretamente colunas como histórico de atraso, valores de fatura e pagamentos, que são centrais para o risco de crédito.

Essa etapa evita análises superficiais e garante que as transformações e novas variáveis tenham sentido de negócio.
Além disso, ajuda a conectar os dados com o problema real, tornando o modelo mais interpretável e confiável.


In [ ]:
# =========================================================
# 4. RENOMEAÇÃO DAS COLUNAS
# =========================================================

df.columns = [
    "limite_credito",
    "sexo",
    "educacao",
    "estado_civil",
    "idade",
    "pagamento_set",
    "pagamento_ago",
    "pagamento_jul",
    "pagamento_jun",
    "pagamento_mai",
    "pagamento_abr",
    "fatura_set",
    "fatura_ago",
    "fatura_jul",
    "fatura_jun",
    "fatura_mai",
    "fatura_abr",
    "pagamento_valor_set",
    "pagamento_valor_ago",
    "pagamento_valor_jul",
    "pagamento_valor_jun",
    "pagamento_valor_mai",
    "pagamento_valor_abr",
    "inadimplente"
]

print("\nColunas renomeadas:")
print(df.columns.tolist())

# Padronização dos nomes

A renomeação das colunas deixou a base mais legível e
aproximou a análise do contexto real do negócio.

Esse ajuste melhora tanto a interpretação dos gráficos
quanto a leitura das etapas de modelagem.


In [ ]:
# =========================================================
# 5. VISÃO GERAL DO DATASET
# =========================================================

print("\n" + "=" * 70)
print("VISÃO GERAL DO DATASET")
print("=" * 70)

print(f"\nShape do dataset: {df.shape}")

print("\nPrimeiras linhas:")
print(df.head(10))

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores nulos por coluna:")
print(df.isnull().sum())

print("\nEstatísticas descritivas:")
print(df.describe())

# Qualidade inicial da base

A base possui 30.000 registros e 24 colunas, sem valores
nulos, o que indica boa consistência inicial para a
análise.

Ao mesmo tempo, as variáveis monetárias já mostram grande
dispersão e a taxa de inadimplência de 22,12% sugere um
alvo desbalanceado, o que exige cuidado na escolha das
métricas.


In [ ]:
# =========================================================
# 7. PREPARAÇÃO PARA ANÁLISE COMPORTAMENTAL
# =========================================================

# Faixas de idade
df["faixa_idade"] = pd.cut(
    df["idade"],
    bins=[20, 30, 40, 50, 60, 80],
    labels=["20-30", "30-40", "40-50", "50-60", "60+"],
    include_lowest=True
)

# Faixas de limite de crédito
df["faixa_limite"] = pd.qcut(
    df["limite_credito"],
    q=4,
    labels=["baixo", "medio", "alto", "muito_alto"]
)

# Média de atraso
col_pagamentos = [
    "pagamento_set",
    "pagamento_ago",
    "pagamento_jul",
    "pagamento_jun",
    "pagamento_mai",
    "pagamento_abr"
]

df["media_atraso"] = df[col_pagamentos].mean(axis=1)

df["nivel_atraso"] = pd.cut(
    df["media_atraso"],
    bins=[-2, 0, 2, 5, 10],
    labels=["sem_atraso", "leve", "moderado", "grave"],
    include_lowest=True
)

# Relação entre fatura média e pagamento médio
faturas = [
    "fatura_set", "fatura_ago", "fatura_jul",
    "fatura_jun", "fatura_mai", "fatura_abr"
]

pagamentos = [
    "pagamento_valor_set", "pagamento_valor_ago", "pagamento_valor_jul",
    "pagamento_valor_jun", "pagamento_valor_mai", "pagamento_valor_abr"
]

df["media_fatura"] = df[faturas].mean(axis=1)
df["media_pagamento"] = df[pagamentos].mean(axis=1)

df["taxa_pagamento"] = df["media_pagamento"] / (df["media_fatura"] + 1)

df["perfil_pagamento"] = pd.qcut(
    df["taxa_pagamento"],
    q=4,
    labels=["baixo_pagamento", "medio_baixo", "medio_alto", "alto_pagamento"]
)

# Leitura comportamental do problema

Nesta etapa, o foco foi entender como os clientes se comportam nos dados.

A ideia foi analisar quem atrasa mais, em quais perfis o risco aumenta e quais variáveis realmente diferenciam bons e maus pagadores.

Isso ajuda a entender melhor os padrões que fazem sentido no contexto de crédito.

In [ ]:
# =========================================================
# 8. FUNÇÃO AUXILIAR PARA TABELAS DE ANÁLISE
# =========================================================

def resumo_inadimplencia(coluna):
    tabela = (
        df.groupby(coluna)["inadimplente"]
        .agg(["count", "mean"])
        .rename(columns={"count": "quantidade", "mean": "taxa_inadimplencia"})
        .sort_values("taxa_inadimplencia", ascending=False)
    )
    tabela["taxa_%"] = (tabela["taxa_inadimplencia"] * 100).round(2)
    return tabela

# Apoio para as análises

A função auxiliar foi criada para padronizar tabelas e
facilitar comparações entre grupos.

In [ ]:
# =========================================================
# 9. ANÁLISE COMPORTAMENTAL - TABELAS
# =========================================================

print("\n" + "=" * 70)
print("TAXA GERAL DE INADIMPLÊNCIA")
print("=" * 70)
print(f"Taxa geral de inadimplência: {df['inadimplente'].mean():.2%}")

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR SEXO")
print("=" * 70)
print(resumo_inadimplencia("sexo"))

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR EDUCAÇÃO")
print("=" * 70)
print(resumo_inadimplencia("educacao"))

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR ESTADO CIVIL")
print("=" * 70)
print(resumo_inadimplencia("estado_civil"))

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR FAIXA DE IDADE")
print("=" * 70)
print(resumo_inadimplencia("faixa_idade"))

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR FAIXA DE LIMITE")
print("=" * 70)
print(resumo_inadimplencia("faixa_limite"))

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR NÍVEL DE ATRASO")
print("=" * 70)
print(resumo_inadimplencia("nivel_atraso"))

print("\n" + "=" * 70)
print("INADIMPLÊNCIA POR PERFIL DE PAGAMENTO")
print("=" * 70)
print(resumo_inadimplencia("perfil_pagamento"))

In [ ]:
# =========================================================
# 10. ANÁLISE COMPORTAMENTAL - GRÁFICOS
# =========================================================

# Distribuição da Inadimplência
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="inadimplente")
plt.title("Distribuição da Inadimplência")
plt.xlabel("Inadimplente")
plt.ylabel("Quantidade")
plt.show()

# Taxa de Inadimplência por Faixa Etária
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="faixa_idade", y="inadimplente", estimator=np.mean, errorbar=None)
plt.title("Taxa de Inadimplência por Faixa Etária")
plt.xlabel("Faixa etária")
plt.ylabel("Taxa de inadimplência")
plt.show()

# Taxa de Inadimplência por Faixa de Limite de Crédito
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="faixa_limite", y="inadimplente", estimator=np.mean, errorbar=None)
plt.title("Taxa de Inadimplência por Faixa de Limite de Crédito")
plt.xlabel("Faixa de limite")
plt.ylabel("Taxa de inadimplência")
plt.show()

# Taxa de Inadimplência por Nível Médio de Atraso
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="nivel_atraso", y="inadimplente", estimator=np.mean, errorbar=None)
plt.title("Taxa de Inadimplência por Nível Médio de Atraso")
plt.xlabel("Nível de atraso")
plt.ylabel("Taxa de inadimplência")
plt.show()

# Taxa de Inadimplência por Perfil de Pagamento
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="perfil_pagamento", y="inadimplente", estimator=np.mean, errorbar=None)
plt.title("Taxa de Inadimplência por Perfil de Pagamento")
plt.xlabel("Perfil de pagamento")
plt.ylabel("Taxa de inadimplência")
plt.xticks(rotation=15)
plt.show()

# Principais achados da análise exploratória

Os resultados mostram que o histórico de atraso é o sinal
mais forte de risco. Clientes sem atraso têm inadimplência
de 13,81%, enquanto perfis com atraso moderado e grave
superam 69%.

Também aparece um padrão financeiro importante: clientes
com limite baixo apresentam inadimplência de 31,79%,
contra 13,98% nos limites muito altos.

No perfil de pagamento, quanto menor a capacidade de pagar
as faturas, maior a chance de default. Isso reforça que o
problema é fortemente comportamental e não apenas
demográfico.


In [ ]:
# =========================================================
# 11. CORRELAÇÃO COM A VARIÁVEL ALVO
# =========================================================

corr = df.corr(numeric_only=True)["inadimplente"].sort_values(ascending=False)

print("\n" + "=" * 70)
print("CORRELAÇÃO COM A INADIMPLÊNCIA")
print("=" * 70)
print(corr)

top_corr = corr.drop("inadimplente").abs().sort_values(ascending=False).head(10).index.tolist()
colunas_heatmap = top_corr + ["inadimplente"]

plt.figure(figsize=(10, 6))
sns.heatmap(df[colunas_heatmap].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heatmap das Variáveis Mais Relacionadas com a Inadimplência")
plt.show()

# Correlação com a variável alvo

A correlação confirma o que apareceu nos gráficos:
pagamento_set e media_atraso estão entre os sinais mais
associados à inadimplência.

Já limite_credito e médias de pagamento aparecem com
relação negativa, sugerindo que maior capacidade
financeira e maior volume pago reduzem o risco.

Mesmo sem indicar causalidade, essa etapa ajuda a
priorizar variáveis para a modelagem.


In [ ]:
# =========================================================
# 12. TRATAMENTO DE DADOS
# =========================================================

df_tratado = df.copy()

print("\n" + "=" * 70)
print("INÍCIO DO TRATAMENTO DE DADOS")
print("=" * 70)

# Variáveis categóricas
colunas_categoricas = ["sexo", "educacao", "estado_civil"]

for col in colunas_categoricas:
    df_tratado[col] = df_tratado[col].astype("category")

print("\nVariáveis categóricas convertidas com sucesso:")
print(df_tratado[colunas_categoricas].dtypes)

# Revisão de categorias inconsistentes
print("\nValores originais de educacao:")
print(df_tratado["educacao"].value_counts().sort_index())

print("\nValores originais de estado_civil:")
print(df_tratado["estado_civil"].value_counts().sort_index())

df_tratado["educacao"] = df_tratado["educacao"].replace({0: 4, 5: 4, 6: 4})
df_tratado["estado_civil"] = df_tratado["estado_civil"].replace({0: 3})

df_tratado["educacao"] = df_tratado["educacao"].astype("category")
df_tratado["estado_civil"] = df_tratado["estado_civil"].astype("category")

print("\nValores ajustados de educacao:")
print(df_tratado["educacao"].value_counts().sort_index())

print("\nValores ajustados de estado_civil:")
print(df_tratado["estado_civil"].value_counts().sort_index())

# Tratamento de outliers nas variáveis monetárias
colunas_monetarias = [
    "limite_credito",
    "fatura_set", "fatura_ago", "fatura_jul",
    "fatura_jun", "fatura_mai", "fatura_abr",
    "pagamento_valor_set", "pagamento_valor_ago", "pagamento_valor_jul",
    "pagamento_valor_jun", "pagamento_valor_mai", "pagamento_valor_abr"
]

limites_outliers = {}

for col in colunas_monetarias:
    q1 = df_tratado[col].quantile(0.25)
    q3 = df_tratado[col].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    limites_outliers[col] = (limite_inferior, limite_superior)

    df_tratado[col] = np.clip(df_tratado[col], limite_inferior, limite_superior)

print("\nOutliers tratados com clipping nas variáveis monetárias.")

# Verificação de valores negativos nas faturas
colunas_faturas = ["fatura_set", "fatura_ago", "fatura_jul", "fatura_jun", "fatura_mai", "fatura_abr"]

print("\nQuantidade de valores negativos nas faturas:")
for col in colunas_faturas:
    qtd_negativos = (df_tratado[col] < 0).sum()
    print(f"{col}: {qtd_negativos}")

# Recriação das variáveis comportamentais após tratamento
df_tratado["media_atraso"] = df_tratado[col_pagamentos].mean(axis=1)
df_tratado["media_fatura"] = df_tratado[faturas].mean(axis=1)
df_tratado["media_pagamento"] = df_tratado[pagamentos].mean(axis=1)
df_tratado["taxa_pagamento"] = df_tratado["media_pagamento"] / (df_tratado["media_fatura"].abs() + 1)

df_tratado["faixa_idade"] = pd.cut(
    df_tratado["idade"],
    bins=[20, 30, 40, 50, 60, 80],
    labels=["20-30", "30-40", "40-50", "50-60", "60+"],
    include_lowest=True
)

df_tratado["faixa_limite"] = pd.qcut(
    df_tratado["limite_credito"],
    q=4,
    labels=["baixo", "medio", "alto", "muito_alto"]
)

df_tratado["nivel_atraso"] = pd.cut(
    df_tratado["media_atraso"],
    bins=[-2, 0, 2, 5, 10],
    labels=["sem_atraso", "leve", "moderado", "grave"],
    include_lowest=True
)

df_tratado["perfil_pagamento"] = pd.qcut(
    df_tratado["taxa_pagamento"],
    q=4,
    labels=["baixo_pagamento", "medio_baixo", "medio_alto", "alto_pagamento"]
)

In [ ]:
# =========================================================
# 13. Visualização do efeito do tratamento
# =========================================================

# Limite de crédito - antes e após tratamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=df["limite_credito"], ax=axes[0])
axes[0].set_title("Limite de crédito - antes do tratamento")
sns.boxplot(y=df_tratado["limite_credito"], ax=axes[1])
axes[1].set_title("Limite de crédito - após tratamento")
plt.tight_layout()
plt.show()

# Pagamento setembro - antes e após tratamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=df["pagamento_valor_set"], ax=axes[0])
axes[0].set_title("Pagamento setembro - antes do tratamento")
sns.boxplot(y=df_tratado["pagamento_valor_set"], ax=axes[1])
axes[1].set_title("Pagamento setembro - após tratamento")
plt.tight_layout()
plt.show()

# Tratamento de dados e engenharia de atributos

O tratamento buscou equilibrar qualidade estatística e
sentido de negócio.

Categorias raras ou inconsistentes foram agrupadas,
outliers monetários foram suavizados com clipping e
valores negativos de fatura foram mantidos por poderem
representar ajustes reais.

Além disso, variáveis derivadas como media_atraso,
media_fatura, media_pagamento e taxa_pagamento ajudam o
modelo a captar padrões mais estáveis do comportamento do
cliente.


In [ ]:
# =========================================================
# 14. PREPARAÇÃO PARA MODELAGEM
# =========================================================

base_modelo = df_tratado.copy()

# Remover colunas criadas apenas para análise visual
colunas_remover = [
    "faixa_idade",
    "faixa_limite",
    "nivel_atraso",
    "perfil_pagamento"
]

base_modelo = base_modelo.drop(columns=colunas_remover)

X = base_modelo.drop("inadimplente", axis=1)
y = base_modelo["inadimplente"]

print("\n" + "=" * 70)
print("BASE FINAL PARA MODELAGEM")
print("=" * 70)
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

# Definição de colunas numéricas e categóricas
colunas_categoricas_modelo = X.select_dtypes(include=["category", "object"]).columns.tolist()
colunas_numericas_modelo = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("\nColunas categóricas:")
print(colunas_categoricas_modelo)

print("\nColunas numéricas:")
print(colunas_numericas_modelo)

# Split treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 70)
print("DIVISÃO TREINO / TESTE")
print("=" * 70)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# Pré-processamento
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", transformador_numerico, colunas_numericas_modelo),
    ("cat", transformador_categorico, colunas_categoricas_modelo)
])

# Preparação para a modelagem

Após o tratamento, a base final passou a ter 27 variáveis
preditoras.

A separação entre colunas categóricas e numéricas, junto
do pipeline de pré-processamento, reduz vazamento de
informação e deixa o fluxo pronto para treino e teste.

A divisão estratificada também preserva a proporção da
classe alvo nas duas amostras.


In [ ]:
# =========================================================
# 15. DEFINIÇÃO DOS MODELOS
# =========================================================

modelos = {
    "Regressão Logística": LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced"
    ),
    "Árvore de Decisão": DecisionTreeClassifier(
        random_state=42,
        class_weight="balanced",
        max_depth=6,
        min_samples_split=20,
        min_samples_leaf=10
    )
}

if xgb_disponivel:
    proporcao_positiva = y_train.value_counts()[0] / y_train.value_counts()[1]

    modelos["XGBoost"] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=proporcao_positiva
    )

# Criar pipelines completos
pipelines = {}
for nome, modelo in modelos.items():
    pipelines[nome] = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", modelo)
    ])

# Escolha dos modelos

Os modelos foram escolhidos para comparar abordagens bem
diferentes no mesmo problema.

A Regressão Logística funciona como baseline
interpretável, o SVM busca fronteiras mais robustas, a
Árvore de Decisão oferece leitura simples e o XGBoost
tende a capturar relações não lineares com maior
eficiência em dados tabulares.


In [ ]:
# =========================================================
# 16. CROSS VALIDATION
# =========================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas_cv = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def avaliar_cross_validation(nome, pipeline, X_train, y_train):
    print("\n" + "=" * 70)
    print(f"CROSS VALIDATION - {nome}")
    print("=" * 70)

    resultados = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=metricas_cv,
        n_jobs=-1
    )

    resumo = {
        "Modelo": nome,
        "CV Accuracy": resultados["test_accuracy"].mean(),
        "CV Precision": resultados["test_precision"].mean(),
        "CV Recall": resultados["test_recall"].mean(),
        "CV F1 Score": resultados["test_f1"].mean(),
        "CV ROC AUC": resultados["test_roc_auc"].mean()
    }

    for metrica, valor in resumo.items():
        if metrica != "Modelo":
            print(f"{metrica}: {valor:.4f}")

    return resumo

resultados_cv = []

for nome, pipeline in pipelines.items():
    resumo_cv = avaliar_cross_validation(nome, pipeline, X_train, y_train)
    resultados_cv.append(resumo_cv)

df_cv = pd.DataFrame(resultados_cv)

print("\n" + "=" * 70)
print("COMPARAÇÃO DOS MODELOS - CROSS VALIDATION")
print("=" * 70)
print(df_cv.sort_values(by="CV ROC AUC", ascending=False))

In [ ]:
# =========================================================
# 17. Gráfico do Cross Validation
# =========================================================

df_cv_plot = df_cv.set_index("Modelo")[["CV Accuracy", "CV Precision", "CV Recall", "CV F1 Score", "CV ROC AUC"]]
df_cv_plot.plot(kind="bar", figsize=(12, 6))
plt.title("Comparação dos Modelos - Cross Validation")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Leitura da validação cruzada

A validação cruzada foi essencial para comparar os modelos
com mais robustez, reduzindo a dependência de uma única
divisão treino-teste.

O SVM teve a maior acurácia média, mas o XGBoost
apresentou o melhor ROC AUC de validação, com 0,7829, além
de recall e F1 competitivos.

Como o alvo é desbalanceado, essa leitura é mais confiável
do que olhar apenas para acurácia.


In [ ]:
# =========================================================
# 18. TREINAMENTO FINAL E AVALIAÇÃO NO TESTE
# =========================================================

def avaliar_modelo_teste(nome_modelo, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_score = pipeline.decision_function(X_test)
        y_proba = (y_score - y_score.min()) / (y_score.max() - y_score.min())

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_proba)

    print("\n" + "=" * 70)
    print(f"RESULTADOS NO TESTE - {nome_modelo}")
    print("=" * 70)
    print(f"Acurácia : {acc:.4f}")
    print(f"Precisão : {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC AUC  : {roc:.4f}")

    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Matriz de Confusão - {nome_modelo}")
    plt.show()

    RocCurveDisplay.from_estimator(pipeline, X_test, y_test)
    plt.title(f"Curva ROC - {nome_modelo}")
    plt.show()

    return {
        "Modelo": nome_modelo,
        "Test Accuracy": acc,
        "Test Precision": prec,
        "Test Recall": rec,
        "Test F1 Score": f1,
        "Test ROC AUC": roc
    }, pipeline

resultados_teste = []
modelos_treinados = {}

for nome, pipeline in pipelines.items():
    resultado_teste, pipeline_treinado = avaliar_modelo_teste(
        nome, pipeline, X_train, X_test, y_train, y_test
    )
    resultados_teste.append(resultado_teste)
    modelos_treinados[nome] = pipeline_treinado

df_teste = pd.DataFrame(resultados_teste)

print("\n" + "=" * 70)
print("COMPARAÇÃO FINAL DOS MODELOS - TESTE")
print("=" * 70)
print(df_teste.sort_values(by="Test ROC AUC", ascending=False))

In [ ]:
# =========================================================
# 19. Gráfico comparativo no teste
# =========================================================

df_teste_plot = df_teste.set_index("Modelo")[["Test Accuracy", "Test Precision", "Test Recall", "Test F1 Score", "Test ROC AUC"]]
df_teste_plot.plot(kind="bar", figsize=(12, 6))
plt.title("Comparação Final dos Modelos - Base de Teste")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Resultados no conjunto de teste

No teste final, o SVM atingiu a maior acurácia, com
0,7780, mas o XGBoost manteve o melhor equilíbrio geral
entre discriminação e sensibilidade para a classe
inadimplente.

O XGBoost entregou recall de 0,6157 e ROC AUC de 0,7771,
superando os demais na capacidade de identificar clientes
de maior risco sem depender só do acerto global.

Já a Regressão Logística teve recall alto, mas perdeu
muito em precisão e acurácia, mostrando menor equilíbrio
no cenário final.


In [ ]:
# =========================================================
# 20. ESCOLHA DO MELHOR MODELO
# =========================================================

melhor_modelo_cv = df_cv.sort_values(by="CV ROC AUC", ascending=False).iloc[0]
melhor_nome = melhor_modelo_cv["Modelo"]

print("\n" + "=" * 70)
print("MELHOR MODELO SEGUNDO CROSS VALIDATION")
print("=" * 70)
print(melhor_modelo_cv)

print("\nMelhor modelo escolhido:", melhor_nome)

# Critério para escolha final

A escolha do melhor modelo foi feita com base
principalmente no desempenho da validação cruzada.

Nesse contexto, o XGBoost foi a decisão mais sólida, pois
apresentou o maior ROC AUC e desempenho consistente também
no teste.

Para um problema de risco de crédito, essa escolha faz
sentido porque separar bem bons e maus pagadores costuma
ser mais importante do que maximizar apenas a acurácia.


In [ ]:
# =========================================================
# 21. IMPORTÂNCIA DAS FEATURES
# =========================================================

pipeline_final = modelos_treinados[melhor_nome]
modelo_final = pipeline_final.named_steps["model"]
preprocessor_final = pipeline_final.named_steps["preprocessor"]

try:
    nomes_features = preprocessor_final.get_feature_names_out()

    if hasattr(modelo_final, "feature_importances_"):
        importancias = modelo_final.feature_importances_

        df_importancias = pd.DataFrame({
            "Feature": nomes_features,
            "Importance": importancias
        }).sort_values("Importance", ascending=False).head(15)

        print("\n" + "=" * 70)
        print(f"TOP 15 FEATURES - {melhor_nome}")
        print("=" * 70)
        print(df_importancias)

        plt.figure(figsize=(10, 6))
        sns.barplot(data=df_importancias, x="Importance", y="Feature")
        plt.title(f"Top 15 Features Mais Importantes - {melhor_nome}")
        plt.xlabel("Importância")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.show()

    elif hasattr(modelo_final, "coef_"):
        coeficientes = np.abs(modelo_final.coef_[0])

        df_coef = pd.DataFrame({
            "Feature": nomes_features,
            "Importance": coeficientes
        }).sort_values("Importance", ascending=False).head(15)

        print("\n" + "=" * 70)
        print(f"TOP 15 FEATURES - {melhor_nome}")
        print("=" * 70)
        print(df_coef)

        plt.figure(figsize=(10, 6))
        sns.barplot(data=df_coef, x="Importance", y="Feature")
        plt.title(f"Top 15 Coeficientes Absolutos - {melhor_nome}")
        plt.xlabel("Magnitude")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.show()

    else:
        print("\nO modelo selecionado não possui medida direta de importância de features.")

except Exception as e:
    print("\nNão foi possível extrair importância das features.")
    print("Detalhe:", e)

# Interpretação das features e conclusão

A análise mostrou que o principal fator de risco está no comportamento recente do cliente.
Variáveis como pagamento_set e media_atraso tiveram maior impacto, indicando que histórico de atraso é o principal driver de inadimplência.

Também foi possível observar que o risco não depende de um único fator, mas da combinação entre comportamento (atrasos e pagamentos)
e capacidade financeira (limite de crédito e valores de fatura).

Em termos de padrões, clientes com atrasos frequentes, baixo volume de pagamento e menor limite tendem a apresentar maior risco.

Na comparação de modelos, o XGBoost apresentou o melhor desempenho geral, principalmente na métrica ROC AUC, mostrando melhor capacidade de separação entre as classes.

Os resultados se mostraram consistentes entre validação cruzada e teste, indicando boa capacidade de generalização do modelo.

Conclusão: a inadimplência pode ser prevista com boa confiabilidade, e os padrões identificados são relevantes para apoiar decisões de crédito,
como concessão, ajuste de limites e priorização de cobrança.
